# Week 2 Assignment: Zombie Detection (PyTorch)

Welcome to this week's programming assignment! You will use torchvision's detection models and retrain [RetinaNet](https://arxiv.org/abs/1708.02002) to spot Zombies using just 5 training images. You will setup the model to restore pretrained weights and fine tune the classification layers.

<img src='https://drive.google.com/uc?export=view&id=18Ck0qNSZy9F1KsUKWc4Jv7_x_1e_fXTN' alt='zombie'>

> This is a PyTorch port of the original TensorFlow Object Detection API assignment. The TensorFlow Model Garden, protobuf configs and `tf.train.Checkpoint` machinery are replaced by `torchvision.models.detection`: the RetinaNet ResNet50-FPN model, its COCO checkpoint (a PyTorch *state dict*) and the classification head class. The exercises follow the same steps as the original. Note that the Coursera autograder expects the TensorFlow results file, so the PyTorch results cannot be submitted for grading.

## Exercises

* [Exercise 1 - Import torchvision detection packages](#exercise-1)
* [Exercise 2 - Visualize the training images](#exercise-2)
* [Exercise 3 - Define the category index dictionary](#exercise-3)
* [Exercise 4 - Download checkpoints](#exercise-4)
* [Exercise 5.1 - Read the model configuration](#exercise-5-1)
* [Exercise 5.2 - Modify the model configuration](#exercise-5-2)
* [Exercise 5.3 - Build the custom model](#exercise-5-3)
* [Exercise 6.1 - Select the weights to restore](#exercise-6-1)
* [Exercise 6.2 - Restore the checkpoint](#exercise-6-2)
* [Exercise 7 - Run a dummy image through the model](#exercise-7)
* [Exercise 8 - Set training hyperparameters](#exercise-8)
* [Exercise 9 - Select the prediction layer variables](#exercise-9)
* [Exercise 10 - Define the training step](#exercise-10)
* [Exercise 11 - Preprocess, predict, and post process an image](#exercise-11)

## Installation

There is nothing to compile or install for this version of the assignment: the detection models, their pre-trained weights and the drawing utilities all ship with `torchvision`, which is already part of the `cv-pytorch` environment.

## Imports

Let's now import the packages you will use in this assignment.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

import os
import random
import zipfile
import io
import urllib.request
import numpy as np

import glob
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from IPython.display import Image as IPyImage

import torch
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

<a name='exercise-1'></a>
### **Exercise 1**: Import torchvision detection packages

Import the necessary modules from the `torchvision` package.
- From [torchvision.models.detection](https://pytorch.org/vision/stable/models.html#object-detection):
  - `retinanet_resnet50_fpn`: This builds your model. You'll use it in place of the Object Detection API's `model_builder`.
  - `RetinaNet_ResNet50_FPN_Weights`: the pre-trained weights (checkpoints) available for this model, together with their metadata (this replaces the `.config` files and `config_util`).
- From [torchvision.models.resnet](https://pytorch.org/vision/stable/models/resnet.html):
  - `ResNet50_Weights`: the ImageNet weights of the backbone.
- From [torchvision.utils](https://pytorch.org/vision/stable/utils.html):
  - `draw_bounding_boxes`: please give this the alias `viz_utils`, as this is what will be used in some visualization code that is given to you later.

In [ ]:
### START CODE HERE (Replace Instances of `None` with your code) ###
# import the model builder and the pre-trained weights
from torchvision.models.detection import retinanet_resnet50_fpn, RetinaNet_ResNet50_FPN_Weights

# import the backbone weights
from torchvision.models.resnet import ResNet50_Weights

# import the utility for drawing boxes. use the alias `viz_utils`
from torchvision.utils import draw_bounding_boxes as viz_utils
### END CODE HERE ###

## Utilities

You'll define a couple of utility functions for loading images and plotting detections. This code is provided for you.

In [ ]:
def load_image_into_numpy_array(path):
    """Load an image from file into a numpy array.

    Puts image into numpy array to feed into the model.
    Note that by convention we put it into a numpy array with shape
    (height, width, channels), where channels=3 for RGB.

    Args:
    path: a file path.

    Returns:
    uint8 numpy array with shape (img_height, img_width, 3)
    """

    image = Image.open(path).convert("RGB")
    (im_width, im_height) = image.size

    return np.array(image.getdata()).reshape(
        (im_height, im_width, 3)).astype(np.uint8)


def plot_detections(image_np,
                    boxes,
                    classes,
                    scores,
                    category_index,
                    figsize=(12, 16),
                    image_name=None,
                    min_score_thresh=0.8):
    """Wrapper function to visualize detections.

    Args:
    image_np: uint8 numpy array with shape (img_height, img_width, 3)
    boxes: a numpy array of shape [N, 4] in normalized [ymin, xmin, ymax, xmax] format
    classes: a numpy array of shape [N]. Note that class indices are 1-based,
          and match the keys in the label map.
    scores: a numpy array of shape [N] or None.  If scores=None, then
          this function assumes that the boxes to be plotted are groundtruth
          boxes and plot all boxes as black with no classes or scores.
    category_index: a dict containing category dictionaries (each holding
          category index `id` and category name `name`) keyed by category indices.
    figsize: size for the figure.
    image_name: a name for the image file.
    """

    image = torch.from_numpy(image_np.copy()).permute(2, 0, 1)   # (3, height, width) uint8
    height, width = image.shape[1], image.shape[2]

    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    keep = np.arange(len(boxes)) if scores is None else np.where(np.asarray(scores) >= min_score_thresh)[0]

    if len(keep) > 0:
        # convert the normalized [ymin, xmin, ymax, xmax] boxes to pixel [xmin, ymin, xmax, ymax]
        ymin, xmin, ymax, xmax = boxes[keep].T
        pixel_boxes = torch.tensor(np.stack([xmin * width, ymin * height, xmax * width, ymax * height], axis=1))

        if scores is None:
            labels, colors = None, "black"
        else:
            labels = [f"{category_index[int(c)]['name']}: {int(100 * s)}%" for c, s in zip(np.asarray(classes)[keep], np.asarray(scores)[keep])]
            colors = "lime"

        font_path = os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data/fonts/ttf/DejaVuSans.ttf")
        image = viz_utils(image, pixel_boxes, labels=labels, colors=colors, width=4, font=font_path, font_size=24)

    image_np_with_annotations = image.permute(1, 2, 0).numpy()

    if image_name:
        plt.imsave(image_name, image_np_with_annotations)

    else:
        plt.imshow(image_np_with_annotations)

## Download the Zombie data

Now you will get 5 images of zombies that you will use for training.
- The zombies are hosted in a Google bucket.
- You can download and unzip the images into a local `data/training/` directory by running the cell below.

In [ ]:
# download the images
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/training-zombie.zip"):
    urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow-3-public/datasets/training-zombie.zip", "data/training-zombie.zip")

# unzip to a local directory
local_zip = 'data/training-zombie.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('data/training')
zip_ref.close()

<a name='exercise-2'></a>

### **Exercise 2**: Visualize the training images

Next, you'll want to inspect the images that you just downloaded.

* Please replace instances of `None` below to load and visualize the 5 training images.
* You can inspect the *training* directory to see the filenames of the zombie images. The paths for the images will look like this:

```
data/training/training-zombie1.jpg
data/training/training-zombie2.jpg
data/training/training-zombie3.jpg
data/training/training-zombie4.jpg
data/training/training-zombie5.jpg
```
- To set file paths, you'll use [os.path.join](https://www.geeksforgeeks.org/python-os-path-join-method/).  As an example, if you wanted to create the path './parent_folder/file_name1.txt', you could write:

`os.path.join('parent_folder', 'file_name' + str(1) + '.txt')`

* You should see the 5 training images after running this cell. If not, please inspect your code, particularly the `image_path`.

In [ ]:
%matplotlib inline

### START CODE HERE (Replace Instances of `None` with your code) ###

# assign the name (string) of the directory containing the training images
train_image_dir = 'data/training'

# declare an empty list
train_images_np = []

# run a for loop for each image
for i in range(1, 6):

    # define the path (string) for each image
    image_path = os.path.join(train_image_dir, 'training-zombie' + str(i) + '.jpg')
    print(image_path)

    # load images into numpy arrays and append to a list
    train_images_np.append(load_image_into_numpy_array(image_path))
### END CODE HERE ###

# configure plot settings via rcParams
plt.rcParams['axes.grid'] = False
plt.rcParams['xtick.labelsize'] = False
plt.rcParams['ytick.labelsize'] = False
plt.rcParams['xtick.top'] = False
plt.rcParams['xtick.bottom'] = False
plt.rcParams['ytick.left'] = False
plt.rcParams['ytick.right'] = False
plt.rcParams['figure.figsize'] = [14, 7]

# plot images
for idx, train_image_np in enumerate(train_images_np):
    plt.subplot(1, 5, idx+1)
    plt.imshow(train_image_np)

plt.show()

<a name='gt_boxes_definition'></a>
## Prepare data for training

In this section, you will create your ground truth boxes. The original Colab notebook offered an interactive annotation tool (`colab_utils.annotate`) to draw your own boxes; that tool only works inside Google Colab, so here you will use the prepopulated list of coordinates provided below. The boxes are in the normalized `[ymin, xmin, ymax, xmax]` format.

If you want to draw your own boxes, any annotation tool will do (e.g. [LabelMe](https://github.com/wkentaro/labelme)); just fill `gt_boxes` with one `[[ymin, xmin, ymax, xmax]]` array per image in the same order as the training images.

In [ ]:
# Define the list of ground truth boxes
gt_boxes = []

<a name='gt-boxes'></a>
#### Use the given ground truth boxes
Run the cell below to use the prepopulated list of coordinates.

In [ ]:
# set this to `True` if you want to override boxes you defined yourself above
override = False

# bounding boxes for each of the 5 zombies found in each image.
ref_gt_boxes = [
        np.array([[0.27333333, 0.41500586, 0.74333333, 0.57678781]]),
        np.array([[0.29833333, 0.45955451, 0.75666667, 0.61078546]]),
        np.array([[0.40833333, 0.18288394, 0.945, 0.34818288]]),
        np.array([[0.16166667, 0.61899179, 0.8, 0.91910903]]),
        np.array([[0.28833333, 0.12543962, 0.835, 0.35052755]]),
      ]

# if gt_boxes is empty, use the reference
if not gt_boxes or override is True:
  gt_boxes = ref_gt_boxes

# if gt_boxes does not contain 5 box coordinates, use the reference
for gt_box in gt_boxes:
    try:
      assert(gt_box is not None)

    except:
      gt_boxes = ref_gt_boxes

      break

#### View your ground truth box coordinates
Please check your list of ground truth box coordinates.

In [ ]:
# print the coordinates of your ground truth boxes
for gt_box in gt_boxes:
  print(gt_box)

Below, we add the class annotations. For simplicity, we assume just a single class, though it should be straightforward to extend this to handle multiple classes. We will also convert everything to the format that the training loop expects (e.g., conversion to tensors, target dictionaries, etc.).

<a name='exercise-3'></a>

### **Exercise 3**: Define the category index dictionary

You'll need to tell the model which integer class ID to assign to the 'zombie' category, and what 'name' to associate with that integer id.

- zombie_class_id: By convention, class ID integers start numbering from 1,2,3, onward.
  - torchvision's detection models reserve the integer 0 for the 'background' class, so the first real class gets the integer 1.
  - Since you are just predicting one class (zombie), please assign `1` to the zombie class ID.

- category_index: Please define the `category_index` dictionary, which will have the same structure as this:
```
{human_class_id :
  {'id'  : human_class_id,
   'name': 'human_so_far'}
}
```
  - Define `category_index` similar to the example dictionary above, except for zombies.
  - This will be used by the succeeding functions to know the class `id` and `name` of zombie images.

- num_classes: In torchvision the number of classes *includes* the background class (index 0). Since you are predicting one class plus the background, please assign `2` to the number of classes that the model will predict.
  - This will be used when you configure the model.

In [ ]:
### START CODE HERE (Replace instances of `None` with your code ###

# Assign the zombie class ID. torchvision reserves 0 for the background,
# so the first real class is 1.
zombie_class_id = 1

# define a dictionary describing the zombie class
category_index = {zombie_class_id: {'id': zombie_class_id, 'name': 'zombie'}}

# Specify the number of classes that the model will predict (including the background)
num_classes = 2
### END CODE HERE ###

In [ ]:
# TEST CODE:

print(category_index[zombie_class_id])
print(num_classes)

**Expected Output:**

```txt
{'id': 1, 'name': 'zombie'}
2
```

### Data preprocessing
You will now do some data preprocessing so it is formatted properly before it is fed to the model:
- convert the train images to float tensors of shape `(3, height, width)` with values in `[0, 1]` (torchvision detectors take a *list* of such tensors, one per image, so images of different sizes are fine).
- convert the ground truth boxes to *pixel* coordinates in the `[xmin, ymin, xmax, ymax]` order torchvision uses, and the class labels to integer tensors.
- torchvision expects one *target dictionary* per image with the keys `boxes` and `labels`.

This code is provided for you.

In [ ]:
train_image_tensors = []

# lists containing the class labels, ground truth boxes and the target dictionaries
gt_classes_tensors = []
gt_box_tensors = []
train_targets = []

for (train_image_np, gt_box_np) in zip(train_images_np, gt_boxes):

    # convert training image to a (3, height, width) float tensor in [0, 1] and add to list
    image_tensor = torch.from_numpy(train_image_np).permute(2, 0, 1).float() / 255.0
    train_image_tensors.append(image_tensor)

    # convert the normalized [ymin, xmin, ymax, xmax] boxes to pixel [xmin, ymin, xmax, ymax]
    height, width = train_image_np.shape[0], train_image_np.shape[1]
    ymin, xmin, ymax, xmax = np.asarray(gt_box_np, dtype=np.float32).T
    boxes = torch.tensor(np.stack([xmin * width, ymin * height, xmax * width, ymax * height], axis=1), dtype=torch.float32)
    gt_box_tensors.append(boxes)

    # the class label of every box (all zombies)
    labels = torch.full((gt_box_np.shape[0],), zombie_class_id, dtype=torch.int64)
    gt_classes_tensors.append(labels)

    # the target dictionary torchvision expects
    train_targets.append({'boxes': boxes, 'labels': labels})

print('Done prepping data.')

## Visualize the zombies with their ground truth bounding boxes

You should see the 5 training images with the bounding boxes after running the cell below. If not, please check your `gt_boxes` list or use the prepopulated `gt_boxes` array given.

In [ ]:
# give boxes a score of 100%
dummy_scores = np.array([1.0], dtype=np.float32)

# define the figure size
plt.figure(figsize=(30, 15))

# use the `plot_detections()` utility function to draw the ground truth boxes
for idx in range(5):
    plt.subplot(2, 4, idx+1)
    plot_detections(
      train_images_np[idx],
      gt_boxes[idx],
      np.ones(shape=[gt_boxes[idx].shape[0]], dtype=np.int32),
      dummy_scores, category_index)

plt.show()

## Download the checkpoint containing the pre-trained weights

Next, you will download the weights of [RetinaNet](https://arxiv.org/abs/1708.02002) trained on COCO.

When working with models that are at the frontiers of research, the models and checkpoints may not yet be organized in a central location. torchvision's "model zoo" is organized around *weights enums*: every pre-trained checkpoint of a model is a member of a `<Model>_Weights` enum.
- It's good practice to do some of this "detective work", so please try the following steps:
  - Go to the [torchvision object detection models page](https://pytorch.org/vision/stable/models.html#object-detection).
  - Find RetinaNet and open the page of `retinanet_resnet50_fpn`.
  - Look for the name of the COCO weights enum member and for the method of a weights enum that downloads its checkpoint as a *state dict*.
- Try to fill out the following code cell below, which does the following:
  - Select the COCO weights of RetinaNet ResNet50 FPN.
  - Download the checkpoint (the state dict); torchvision caches it under `~/.cache/torch/hub/checkpoints/`.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>The weights enum you imported in Exercise 1 has a member for the COCO checkpoint (its name ends with <code>_V1</code>).</li>
    <li>Every weights enum member has a <code>get_state_dict(progress=True)</code> method that downloads the checkpoint file and returns the state dict, and a <code>.url</code> attribute if you want to see where it comes from.</li>
</ul>
</p>

<a name='exercise-4'></a>
### Exercise 4: Download checkpoints

  - Select the COCO weights of RetinaNet ResNet50 FPN.
  - Download the checkpoint (state dict).

In [ ]:
### START CODE HERE ###
# select the COCO weights of RetinaNet ResNet50 FPN
weights = RetinaNet_ResNet50_FPN_Weights.COCO_V1

# download the checkpoint (state dict). torchvision caches it under ~/.cache/torch
checkpoint_state_dict = weights.get_state_dict(progress=True)

### END CODE HERE

print(weights.url)
print(f"the checkpoint contains {len(checkpoint_state_dict)} tensors")

**Expected Output**:

```txt
https://download.pytorch.org/models/retinanet_resnet50_fpn_coco-eeacb38b.pth
the checkpoint contains 301 tensors
```

## Configure the model
Here, you will configure the model for this use case.

<a name='exercise-5-1'></a>

### **Exercise 5.1**: Read the model configuration

#### configs
The Object Detection API described a model with a `.config` file. In torchvision the equivalent information lives in two places:
- the keyword arguments of the model builder (`num_classes`, `min_size`, `max_size`, ...), and
- the `meta` dictionary of the weights, which records the classes the checkpoint was trained on, the image size it was evaluated at, and more.

Please set `configs` to the `meta` dictionary of the weights you selected.

In [ ]:
### START CODE HERE ###
# Read the metadata of the pre-trained weights
configs = weights.meta

### END CODE HERE ###
# See what configs looks like
print(configs.keys())
print("number of classes:", len(configs['categories']))
print("first classes:", configs['categories'][:5])
print("image size:", configs['min_size'])

**Expected Output** (the exact set of keys may vary between torchvision versions):

```txt
dict_keys(['categories', 'min_size', 'num_params', 'recipe', '_metrics', '_ops', '_file_size', '_docs'])
number of classes: 91
first classes: ['__background__', 'person', 'bicycle', 'car', 'motorcycle']
image size: (1, 1)
```

<a name='exercise-5-2'></a>

### **Exercise 5.2**: Modify the model configuration

Define `model_config`, a dictionary with the keyword arguments you will pass to the model builder:
- `num_classes`: Modify the number of classes from its default of `91` (COCO) to the `num_classes` that you set earlier in this notebook.
- `min_size` and `max_size`: the model resizes every image so that its smaller side is `min_size` (default 800) and its larger side at most `max_size` (default 1333). Set both to `640` so that images are resized to at most 640 pixels, like the `640 x 640` resizer of the original configuration.

You don't need to freeze batch normalization yourself: when the builder is given pre-trained backbone weights it automatically uses `FrozenBatchNorm2d` layers. You will check this after building the model.

In [ ]:
### START CODE HERE ###
model_config = {
    # the number of classes (including the background)
    'num_classes': num_classes,

    # the image size
    'min_size': 640,
    'max_size': 640,
}
### END CODE HERE

# See what model_config now looks like after you've customized it!
model_config

## Build the model

Recall that you imported `retinanet_resnet50_fpn`.  
- You'll use it to build the model according to the configuration that you have just customized.

<a name='exercise-5-3'></a>

### **Exercise 5.3**: Build the custom model

#### retinanet_resnet50_fpn
The builder has the following signature:

```
def retinanet_resnet50_fpn(*, weights=None, progress=True, num_classes=None, weights_backbone=ResNet50_Weights.IMAGENET1K_V1, trainable_backbone_layers=None, **kwargs):

```
- weights: keep the default `None`. You will restore the COCO weights *selectively* in the next section (passing the COCO weights here would also force `num_classes` back to 91).
- weights_backbone: keep the default ImageNet weights of the ResNet50 backbone (`ResNet50_Weights.IMAGENET1K_V1`). This is what switches the batch normalization layers to their frozen version.
- Pass your `model_config` as keyword arguments (`**model_config`).
- Move the model to the `device`.
- Note that it will take some time to build the model the first time (the backbone weights are downloaded).

In [ ]:
### START CODE HERE (Replace instances of `None` with your code) ###
# weights=None because the COCO head predicts 91 classes; you restore the rest of the
# checkpoint selectively below. Keeping the pretrained backbone weights is what switches
# the batch normalization layers to their frozen version.
detection_model = retinanet_resnet50_fpn(
    weights=None,
    weights_backbone=ResNet50_Weights.IMAGENET1K_V1,
    **model_config,
).to(device)
### END CODE HERE ###

print(type(detection_model))

**Expected Output**:

```txt
<class 'torchvision.models.detection.retinanet.RetinaNet'>
```

In [ ]:
# TEST CODE: the batch normalization layers of the backbone should be frozen
print(type(detection_model.backbone.body.bn1))
assert type(detection_model.backbone.body.bn1).__name__ == 'FrozenBatchNorm2d', "batch normalization is not frozen: did you keep the pre-trained backbone weights?"

**Expected Output**:

```txt
<class 'torchvision.ops.misc.FrozenBatchNorm2d'>
```

## Restore weights from your checkpoint

Now, you will selectively restore weights from your checkpoint.
- Your end goal is to create a custom model which reuses parts of, but not all of the layers of RetinaNet (currently stored in the variable `detection_model`.)
  - The parts of RetinaNet that you want to reuse are:
    - Feature extraction layers
    - Bounding box regression prediction layer
  - The part of RetinaNet that you will not want to reuse is the classification prediction layer (since you will define and train your own classification layer specific to zombies).
  - For the parts of RetinaNet that you want to reuse, you will also restore the weights from the checkpoint that you selected.

#### Inspect the detection_model
First, take a look at the top-level parts (children) of the model.

In [ ]:
# Run this to check the parts of detection_model
for name, child in detection_model.named_children():
    print(f"{name}: {type(child).__name__}")

#### Find the source code for detection_model

You'll see that the model is made of a `backbone`, an `anchor_generator`, a `head` and a `transform`.
Please practice some detective work and open up the source code for this class in the torchvision GitHub repository: [retinanet.py](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/retinanet.py).

- `backbone` is the ResNet50 + FPN feature extractor, which you will want to reuse for your zombie detector model.
- `head` is a `RetinaNetHead`, which holds the two prediction branches.

#### Inspect `head`
Now, check the parts of the head.

In [ ]:
for name, child in detection_model.head.named_children():
    print(f"{name}: {type(child).__name__}")
    for sub_name, sub_child in child.named_children():
        print(f"    {sub_name}: {type(sub_child).__name__}")

You'll see that the head contains two branches, each with two parts:

```
classification_head: RetinaNetClassificationHead
    conv: Sequential
    cls_logits: Conv2d
regression_head: RetinaNetRegressionHead
    conv: Sequential
    bbox_reg: Conv2d
```

In the source code of [retinanet.py](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/retinanet.py), look at the `RetinaNetClassificationHead` and `RetinaNetRegressionHead` classes to get a sense for what these parts represent.

#### Inspect `conv`
The `conv` attribute of each branch is a stack of convolution layers that comes *before* the prediction layer. In the original Object Detection API these were called the "tower" layers (`_base_tower_layers_for_heads`). Both towers were trained on COCO and you will want to reuse them in your model.

#### Inspect `bbox_reg`
`detection_model.head.regression_head.bbox_reg` is the bounding box prediction layer (the Object Detection API's `_box_prediction_head`), which you'll want to use for your model.

#### Inspect `cls_logits`
`detection_model.head.classification_head.cls_logits` is the layer that predicts the class (category). Print its shape and compare it with the shape stored in the COCO checkpoint:

In [ ]:
print("model      :", tuple(detection_model.head.classification_head.cls_logits.weight.shape))
print("checkpoint :", tuple(checkpoint_state_dict['head.classification_head.cls_logits.weight'].shape))

The COCO checkpoint predicts 91 classes for each of the 9 anchors (`9 * 91 = 819` output channels) while your model predicts 2 (`9 * 2 = 18`). The shapes don't match, which is one more reason not to restore this layer.

#### Which layers will you reuse?
Remember that you are reusing the model for its feature extraction and bounding box detection.
- You will create your own classification layer and train it on zombie images.
- So you won't need to reuse the class prediction layer of `detection_model`.

## Restore the desired layers
You will now isolate the weights of the checkpoint that you wish to restore, and load them into the model.
- First, select the checkpoint entries for everything except the class prediction layer
- Next, load them into the model with [load_state_dict](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.load_state_dict).

As a reminder of how a state dict works: it is a dictionary mapping parameter names (e.g. `'head.regression_head.bbox_reg.weight'`) to tensors. Every parameter of the model has an entry, and the names follow the attribute path in the model:

```
backbone.body.conv1.weight
...
head.classification_head.conv.0.0.weight
...
head.classification_head.cls_logits.weight
head.classification_head.cls_logits.bias
head.regression_head.conv.0.0.weight
...
head.regression_head.bbox_reg.weight
head.regression_head.bbox_reg.bias
```

`model.load_state_dict(state_dict, strict=False)` copies every entry whose name exists in the model and returns the names that were *missing* from the state dict (and any *unexpected* names). With `strict=True` (the default) any mismatch raises an error.

Try this out step by step!

<a name='exercise-6-1'></a>
### Exercise 6.1: Select the weights to restore

- Please define `tmp_state_dict` as a dictionary containing every entry of `checkpoint_state_dict` **except** the two entries (weight and bias) of the class prediction layer.
  - The names of those two entries start with the prefix stored in `cls_logits_prefix` below.
- Note, you won't include the class prediction layer, but you will include the classification tower (`head.classification_head.conv`).
- Important: Be careful to avoid typos in the prefix. If the prefix does not match anything, the class prediction layer will be included, and loading the state dict will fail with a shape mismatch.

In [ ]:
### START CODE HERE ###

# the prefix of the parameter names of the class prediction layer
cls_logits_prefix = 'head.classification_head.cls_logits'

# every entry of the checkpoint whose name does not start with the prefix
tmp_state_dict = {name: tensor for name, tensor in checkpoint_state_dict.items()
                  if not name.startswith(cls_logits_prefix)}

### END CODE HERE

In [ ]:
# Check the datatype and size of this state dict
print(type(tmp_state_dict))
print(len(checkpoint_state_dict), len(tmp_state_dict))

# Expected output:
# <class 'dict'>
# 301 299

In [ ]:
# Check that the class prediction layer is not included
print([name for name in tmp_state_dict if 'cls_logits' in name])
print([name for name in tmp_state_dict if 'bbox_reg' in name])

#### Expected output
```
[]
['head.regression_head.bbox_reg.weight', 'head.regression_head.bbox_reg.bias']
```

<a name='exercise-6-2'></a>
### Exercise 6.2: Restore the checkpoint

You can now restore the checkpoint.

- Call `load_state_dict()` on the detection model, passing in your temporary state dict.
- **IMPORTANT**: set `strict=False`. Since the class prediction layer is not in your state dict, the strict check would fail. With `strict=False`, `load_state_dict` returns the names of the entries that were not restored, which lets you verify that *only* the class prediction layer is left untouched.
  - If you restore nothing (e.g. by passing an empty dictionary), there won't be any immediate error message, but later during training you'll notice that your model's loss doesn't improve, which means that the pre-trained weights were not restored properly.

In [ ]:
### START CODE HERE ###

# Restore the weights and keep the names that were not restored.
# strict=False is required because the class prediction layer is deliberately absent.
missing_keys, unexpected_keys = detection_model.load_state_dict(tmp_state_dict, strict=False)

### END CODE HERE ###

print("missing keys   :", missing_keys)
print("unexpected keys:", unexpected_keys)

**Expected Output**:

```txt
missing keys   : ['head.classification_head.cls_logits.weight', 'head.classification_head.cls_logits.bias']
unexpected keys: []
```

<a name='exercise-7'></a>
### **Exercise 7**: Run a dummy image through the model

Run a dummy image through the model to check that it works end to end. (Unlike TensorFlow, PyTorch creates all the model variables when the model is built, so this step is not needed to create them; it is still a good sanity check of the shapes and of the output format.)

Recall that `detection_model` is an object of type [torchvision.models.detection.retinanet.RetinaNet](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/retinanet.py)

Important behaviours of the `detection_model` object are:
- `detection_model.transform`:
    - takes a list of image tensors (and optionally targets) and
    - normalizes the images with the ImageNet mean/std, resizes them according to `min_size`/`max_size`, and batches them (padding them to the same size).
    - this is the equivalent of the Object Detection API's `preprocess()` and is called automatically inside the forward pass.

- `detection_model.eval()` followed by `detection_model(list_of_images)`
  - runs the forward pass in *inference* mode: the images are preprocessed, predicted and post-processed (score thresholding, non-maximum suppression, resizing the boxes back to the original image size).
  - returns a list with one Python dictionary per image with the keys `boxes`, `scores` and `labels`.
  - this is the equivalent of `preprocess()` + `predict()` + `postprocess()`.
  - Remember that your images have dimensions 640 x 640 x 3 (in PyTorch's channels-first layout: 3 x 640 x 640).
  - For the dummy image, you can declare a [tensor of zeros](https://pytorch.org/docs/stable/generated/torch.zeros.html), and pass a list containing that single image.
  - Wrap the call in `torch.no_grad()` since no gradients are needed.

**Note**: Please use the recommended variable names, which include the prefix `tmp_`, since these variables won't be used later, but you'll define similarly-named variables later for predicting on actual zombie images.

In [ ]:
### START CODE HERE (Replace instances of `None` with your code)###

# create a dummy image tensor of zeros (channels first) on the device
tmp_image = torch.zeros(3, 640, 640).to(device)

# put the model in evaluation mode
detection_model.eval()

# run the forward pass (inside torch.no_grad()) to get the detections for the (list of) dummy image(s)
with torch.no_grad():
    tmp_detections = detection_model([tmp_image])

### END CODE HERE ###

print('Weights restored!')
print(tmp_detections[0].keys())

In [ ]:
# Test Code:
assert len(list(detection_model.parameters())) > 0, "the model has no parameters"

params = list(detection_model.parameters())
print(tuple(params[0].shape))
print(tuple(detection_model.head.classification_head.cls_logits.weight.shape))
print(tuple(detection_model.head.regression_head.bbox_reg.weight.shape))

**Expected Output**:

```txt
dict_keys(['boxes', 'scores', 'labels'])
(64, 3, 7, 7)
(18, 256, 3, 3)
(36, 256, 3, 3)
```

## Custom training loop

With the data and model now setup, you can now proceed to configure the training.

<a name='exercise-8'></a>
### **Exercise 8**: Set training hyperparameters

Set an appropriate learning rate for the training.

- batch_size: you can use 4
  - You can increase the batch size up to 5, since you have just 5 images for training.
- num_batches: You can use 100
  - You can increase the number of batches but the training will take longer to complete.
- learning_rate: You can use 0.01
  - When you run the training loop later, notice how the initial loss INCREASES` before decreasing.
  - You can try a lower learning rate to see if you can avoid this increased loss.
- The optimizer needs the list of variables it will update, so you will create it in the next exercise, after selecting the layers to fine-tune.

Training will be fairly quick, so we do encourage you to experiment a bit with these hyperparameters!

In [ ]:
### START CODE HERE (Replace instances of `None` with your code)###

# set the batch_size
batch_size = 4

# set the number of batches
num_batches = 100

# Set the learning rate
learning_rate = 0.01

### END CODE HERE ###

## Choose the layers to fine-tune

To make use of transfer learning and pre-trained weights, you will train just certain parts of the detection model, namely, the last prediction layers.
- Please take a minute to inspect the parameters of `detection_model`.

In [ ]:
# Inspect the parameters of detection_model
for i, (name, v) in enumerate(detection_model.named_parameters()):
    print(f"i: {i} \t name: {name} \t shape:{tuple(v.shape)} \t dtype={v.dtype}")

Notice that there are some parameters whose names are prefixed with the following:
```
head.classification_head.conv
...
head.classification_head.cls_logits
...
head.regression_head.conv
...
head.regression_head.bbox_reg
...
```

Among these, which do you think are the prediction layers at the "end" of the model?
- Recall that when inspecting the source code to restore the checkpoints ([retinanet.py](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/retinanet.py)) you noticed that:
  - `conv`: refers to the layers that are placed right before the prediction layer (the "tower")
  - `bbox_reg` refers to the prediction layer for the bounding boxes
  - `cls_logits` refers to the prediction layer for the classification


So you can see that in the source code for this model, `conv` refers to layers that are before the prediction layer, and `cls_logits`/`bbox_reg` refer to the prediction layers.

<a name='exercise-9'></a>

### **Exercise 9**: Select the prediction layer variables

Based on inspecting `detection_model.named_parameters()`, please select the prediction layer variables that you will fine tune:
- The bounding box head variables (which predict bounding box coordinates)
- The class head variables (which predict the class/category)

You have a few options for doing this:
- You can access them by attribute:
```
detection_model.head.regression_head.bbox_reg.weight
```

- Alternatively, you can use string matching to select the variables:
```
tmp_list = []
for name, v in detection_model.named_parameters():
  if name.startswith('backbone.fpn'):
    tmp_list.append(v)
```

**Hint**: There are a total of four variables that you want to fine tune.

Then create the optimizer:
- optimizer: you can use [torch.optim.SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html)
  - Pass in the list of variables to fine tune
  - Set the learning rate
  - Set the momentum to 0.9

In [ ]:
### START CODE HERE (Replace instances of `None` with your code) ###

# define a list that contains the layers that you wish to fine tune:
# the two prediction heads, and nothing before them
to_fine_tune = [
    detection_model.head.regression_head.bbox_reg.weight,
    detection_model.head.regression_head.bbox_reg.bias,
    detection_model.head.classification_head.cls_logits.weight,
    detection_model.head.classification_head.cls_logits.bias,
]

# set the optimizer and pass in the variables to fine tune and the learning_rate
optimizer = torch.optim.SGD(to_fine_tune, lr=learning_rate, momentum=0.9)

### END CODE HERE

In [ ]:
# Test Code:

names_by_param = {id(v): name for name, v in detection_model.named_parameters()}
print(names_by_param[id(to_fine_tune[0])])
print(names_by_param[id(to_fine_tune[2])])

**Expected Output** (the order of the four variables may differ):

```txt
head.regression_head.bbox_reg.weight
head.classification_head.cls_logits.weight
```

Since only these four variables will be updated, the other parameters don't need gradients. Turning them off makes the backward pass much cheaper (this code is provided for you).

In [ ]:
# freeze everything...
for v in detection_model.parameters():
    v.requires_grad_(False)

# ...except the variables you selected
for v in to_fine_tune:
    v.requires_grad_(True)

print("trainable parameters:", sum(v.numel() for v in detection_model.parameters() if v.requires_grad))

## Train your model

You'll define a function that handles training for one batch, which you'll later use in your training loop.

First, walk through these code cells to learn how you'll perform training using this model.

In [ ]:
# Get a batch of your training images and their targets
g_images_list = [image.to(device) for image in train_image_tensors[0:2]]
g_targets_list = [{k: v.to(device) for k, v in target.items()} for target in train_targets[0:2]]

The `detection_model` is of class [RetinaNet](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/retinanet.py), and its source code shows that it holds a `transform` (a [GeneralizedRCNNTransform](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/transform.py)).
- This preprocesses the images so that they can be passed into the network (for training or prediction):
```
  def forward(self, images, targets=None):
    """
    Args:
      images: a list of [channels, height_in, width_in] float tensors representing
        a batch of images with values between 0 and 1.
      targets: an optional list of target dictionaries (boxes are resized together with the images)
    Returns:
      image_list: an ImageList holding the batched, normalized and resized images (`.tensors`)
        and the size of each image inside the batch (`.image_sizes`), as resized images
        can be padded with zeros.
      targets: the resized targets
    """
```

In [ ]:
# Use .transform to preprocess the images
g_preprocessed_images, g_preprocessed_targets = detection_model.transform(g_images_list, g_targets_list)
print(f"g_preprocessed_images type: {type(g_preprocessed_images)}")
print(f"the batched image tensor has shape {g_preprocessed_images.tensors.shape}")
print(f"information about each image's true shape excluding padding: {g_preprocessed_images.image_sizes}")
print(f"the resized boxes of the first image: {g_preprocessed_targets[0]['boxes']}")

## Make a prediction and calculate the loss

You don't need to call the transform yourself: the model's forward pass does it internally. What the forward pass returns depends on the *mode* of the model:
- in evaluation mode (`detection_model.eval()`), it takes the list of images and returns the post-processed detections (you saw this in Exercise 7);
- in training mode (`detection_model.train()`), it takes the list of images **and** the list of targets, and returns a dictionary of losses.

According to the source code, the loss dictionary contains:
- `classification`: the (focal) classification loss
- `bbox_regression`: the localization loss

Notice that in training mode `forward` requires the targets. If you tried to call it without them, you'd get an error.

In [ ]:
# Try to call the model in training mode without targets; look at the error message
detection_model.train()
try:
    detection_model(g_images_list)
except (AssertionError, ValueError) as e:
    print("Error message:", e)

This makes sense, since the loss is comparing the prediction to the ground truth, and so the loss function needs to know the ground truth.
#### Provide the ground truth
The ground truth is passed as the second argument, as a list of dictionaries with:
- The true bounding boxes (`boxes`, in pixels, `[xmin, ymin, xmax, ymax]`)
- The true classes (`labels`)

In [ ]:
# Calculate the losses by providing the ground truth
losses_dict = detection_model(g_images_list, g_targets_list)

# View the loss dictionary
print(f"loss dictionary keys: {losses_dict.keys()}")
print(f"localization loss {losses_dict['bbox_regression'].item():.8f}")
print(f"classification loss {losses_dict['classification'].item():.8f}")

You can now calculate the gradient and optimize the variables that you selected to fine tune.
- Use `loss.backward()` and `optimizer.step()`

```Python
# clear the gradients of the previous step
optimizer.zero_grad()

# Make the prediction and calculate the losses
losses_dict = model(images, targets)

# calculate the gradient of each model variable with respect to the loss
[some loss].backward()

# apply the gradients to update these model variables
optimizer.step()
```

<a name='exercise-10'></a>

### **Exercise 10**: Define the training step
Please complete the function below to set up one training step.
- Put the model in training mode
- Clear the gradients
- Make a prediction and calculate the losses (the forward pass in training mode does both, and the ground truth is passed alongside the images)
- Calculate the total loss:
  - `total_loss` = `localization_loss + classification_loss`
- Calculate gradients with respect to the variables you selected to train (`backward()`)
- Optimize the model's variables (`optimizer.step()`)

In [ ]:
def train_step_fn(image_list,
                groundtruth_boxes_list,
                groundtruth_classes_list,
                model,
                optimizer,
                vars_to_fine_tune,
                device):
    """A single training iteration.

    Args:
      image_list: A list of [3, height, width] Tensors of type torch.float32.
        Note that the height and width can vary across images, as they are
        reshaped within the model to be 640x640.
      groundtruth_boxes_list: A list of Tensors of shape [N_i, 4] with type
        torch.float32 representing groundtruth boxes for each image in the batch.
      groundtruth_classes_list: A list of Tensors of shape [N_i] with type
        torch.int64 representing the groundtruth classes for each image in
        the batch.

    Returns:
      A scalar tensor representing the total loss for the input batch.
    """

    # move the images to the device and build the target dictionaries
    images = [image.to(device) for image in image_list]
    targets = [{'boxes': boxes.to(device), 'labels': classes.to(device)}
               for boxes, classes in zip(groundtruth_boxes_list, groundtruth_classes_list)]

    ### START CODE HERE (Replace instances of `None` with your code) ###

    # Put the model in training mode
    model.train()

    # Clear the gradients
    optimizer.zero_grad()

    # Make a prediction and calculate the losses (provide the ground truth).
    # In training mode the forward pass returns the losses rather than the detections.
    losses_dict = model(images, targets)

    # Calculate the total loss (sum of both losses)
    total_loss = losses_dict['classification'] + losses_dict['bbox_regression']

    # Calculate the gradients
    total_loss.backward()

    # Optimize the model's selected variables
    optimizer.step()

    ### END CODE HERE ###

    return total_loss

## Run the training loop

Run the training loop using the training step function that you just defined.

**Note:** *Each step runs the frozen backbone on four 640x640 images. On a CPU the loop takes a few minutes; it is much faster on a GPU or on Apple's MPS backend.*

In [ ]:
print('Start fine-tuning!', flush=True)

for idx in range(num_batches):
    # Grab keys for a random subset of examples
    all_keys = list(range(len(train_images_np)))
    random.shuffle(all_keys)
    example_keys = all_keys[:batch_size]

    # Get the ground truth
    gt_boxes_list = [gt_box_tensors[key] for key in example_keys]
    gt_classes_list = [gt_classes_tensors[key] for key in example_keys]

    # get the images
    image_tensors = [train_image_tensors[key] for key in example_keys]

    # Training step (forward pass + backwards pass)
    total_loss = train_step_fn(image_tensors,
                               gt_boxes_list,
                               gt_classes_list,
                               detection_model,
                               optimizer,
                               to_fine_tune,
                               device
                              )

    if idx % 10 == 0:
        print('batch ' + str(idx) + ' of ' + str(num_batches)
        + ', loss=' +  str(total_loss.item()), flush=True)

print('Done fine-tuning!')

**Expected Output:**

Total loss should be decreasing and should be well below 1 after fine tuning. It may go up during the first batches before it comes down. For example:

```txt
Start fine-tuning!
batch 0 of 100, loss=1.3794361352920532
batch 10 of 100, loss=0.9008064866065979
batch 20 of 100, loss=0.24269652366638184
batch 30 of 100, loss=0.05844573676586151
batch 40 of 100, loss=0.024996722117066383
batch 50 of 100, loss=0.015768427401781082
batch 60 of 100, loss=0.026593483984470367
batch 70 of 100, loss=0.02110370807349682
batch 80 of 100, loss=0.024027537554502487
batch 90 of 100, loss=0.023314040154218674
Done fine-tuning!
```

## Load test images and run inference with new model!

You can now test your model on a new set of images. The cell below downloads 237 images of a walking zombie and stores them in a `results/` directory.

In [ ]:
# download test images
if not os.path.exists("data/zombie-walk-frames.zip"):
    urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow-3-public/datasets/zombie-walk-frames.zip", "data/zombie-walk-frames.zip")

# unzip test images
local_zip = 'data/zombie-walk-frames.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('./results')
zip_ref.close()

You will load these images into numpy arrays to prepare it for inference.

In [ ]:
test_image_dir = './results/'
test_images_np = []

# load images into a numpy array. this will take a few minutes to complete.
for i in range(0, 237):
    image_path = os.path.join(test_image_dir, 'zombie-walk' + "{0:04}".format(i) + '.jpg')
    print(image_path)
    test_images_np.append(np.expand_dims(
      load_image_into_numpy_array(image_path), axis=0))

<a name='exercise-11'></a>

### **Exercise 11**: Preprocess, predict, and post process an image

Define a function that returns the detection boxes, classes, and scores.

In [ ]:
def detect(input_tensor, detection_model):
    """Run detection on an input image.

    Args:
    input_tensor: A [3, height, width] Tensor of type torch.float32 with values in [0, 1].
      Note that height and width can be anything since the image will be
      immediately resized according to the needs of the model within this
      function.

      detection_model: the fine-tuned RetinaNet, passed in rather than read from the
        surrounding notebook scope.

    Returns:
    A dict containing 3 numpy arrays (`detection_boxes` in normalized [ymin, xmin, ymax, xmax] format,
      `detection_classes`, and `detection_scores`), sorted by decreasing score.
    """

    ### START CODE HERE (Replace instances of `None` with your code) ###
    # put the model in evaluation mode
    detection_model.eval()

    # run the model on the (list of) image(s) inside torch.no_grad() to get the final detections
    # (the model preprocesses the image, predicts and post-processes the predictions)
    with torch.no_grad():
        detections = detection_model([input_tensor])[0]
    ### END CODE HERE ###

    # convert the pixel [xmin, ymin, xmax, ymax] boxes back to normalized [ymin, xmin, ymax, xmax]
    height, width = input_tensor.shape[1], input_tensor.shape[2]
    boxes = detections['boxes'].cpu().numpy()
    if len(boxes) == 0:
        boxes = np.zeros((1, 4), dtype=np.float32)
        scores = np.zeros((1,), dtype=np.float32)
        classes = np.zeros((1,), dtype=np.int64)
    else:
        xmin, ymin, xmax, ymax = boxes.T
        boxes = np.stack([ymin / height, xmin / width, ymax / height, xmax / width], axis=1)
        scores = detections['scores'].cpu().numpy()
        classes = detections['labels'].cpu().numpy()

    return {'detection_boxes': boxes, 'detection_classes': classes, 'detection_scores': scores}

You can now loop through the test images and get the detection scores and bounding boxes to overlay in the original image. We will save each result in a `results` dictionary (in the original course the autograder used this to evaluate your results).

In [ ]:
results = {'boxes': [], 'scores': []}

for i in range(len(test_images_np)):
    print('Running inference on %d of %d images...' % (i, len(test_images_np)))
    input_tensor = torch.from_numpy(test_images_np[i][0]).permute(2, 0, 1).float().to(device) / 255.0
    detections = detect(input_tensor, detection_model)
    plot_detections(
      test_images_np[i][0],
      detections['detection_boxes'],
      detections['detection_classes'].astype(np.uint32),
      detections['detection_scores'],
      category_index, figsize=(15, 20), image_name="./results/gif_frame_" + ('%03d' % i) + ".jpg")
    results['boxes'].append(detections['detection_boxes'][0])
    results['scores'].append(detections['detection_scores'][0])

In [ ]:
# TEST CODE

print(len(results['boxes']))
print(results['boxes'][0].shape)
print()

# compare with expected bounding boxes
print(np.allclose(results['boxes'][0], [0.28838485, 0.06830047, 0.7213766 , 0.19833465], rtol=0.18))
print(np.allclose(results['boxes'][5], [0.29168868, 0.07529271, 0.72504973, 0.20099735], rtol=0.18))
print(np.allclose(results['boxes'][10], [0.29548776, 0.07994056, 0.7238164 , 0.20778716], rtol=0.18))

**Expected Output:** Ideally the three boolean values at the bottom should be `True`. This compares your resulting bounding boxes for each zombie image to some preloaded coordinates (i.e. the hardcoded values in the test cell above, which come from the original TensorFlow model). Depending on how you annotated the training images and on the model, it's possible that some of your results differ for these three frames but still get good results overall when all images are examined. If two or all are False, please try annotating the images again with a tighter bounding box or use the [predefined `gt_boxes` list](#gt-boxes).

```txt
237
(4,)

True
True
True
```

You can also check if the model detects a zombie class in the images by examining the `scores` key of the `results` dictionary. You should get higher than 88.0 here.

In [ ]:
x = np.array(results['scores'])

# percent of frames where a zombie is detected
zombie_detected = (np.where(x > 0.9, 1, 0).sum())/237*100
print(zombie_detected)

You can also display some still frames and inspect visually. If you don't see a bounding box around the zombie, please consider re-annotating the ground truth or use the predefined `gt_boxes` [here](#gt-boxes)

In [ ]:
print('Frame 0')
display(IPyImage('./results/gif_frame_000.jpg'))
print()
print('Frame 5')
display(IPyImage('./results/gif_frame_005.jpg'))
print()
print('Frame 10')
display(IPyImage('./results/gif_frame_010.jpg'))

## Create a zip of the zombie-walk images.
You can download this if you like to create your own animations

In [ ]:
zipf = zipfile.ZipFile('./zombie.zip', 'w', zipfile.ZIP_DEFLATED)

filenames = glob.glob('./results/gif_frame_*.jpg')
filenames = sorted(filenames)

for filename in filenames:
    zipf.write(filename)

zipf.close()

## Create Zombie animation

In [ ]:
anim_file = './zombie-anim.gif'

filenames = glob.glob('./results/gif_frame_*.jpg')
filenames = sorted(filenames)
images = []

for filename in filenames:
    image = Image.open(filename)
    images.append(image)

# save an animated GIF at 10 frames per second
images[0].save(anim_file, save_all=True, append_images=images[1:], duration=100, loop=0)
print(f"saved {anim_file}")

Displaying the large `gif` inside the notebook can be slow. To view the animation, open `zombie-anim.gif` from your file browser instead.

## Save results file

Run the cell below to save your results. (In the original course, `results.data` was uploaded to the Coursera grader.)

In [ ]:
import pickle

# remove file if it exists
if os.path.exists('results.data'):
    os.remove('results.data')

# write results to binary file.
with open('results.data', 'wb') as filehandle:
    pickle.dump(results['boxes'], filehandle)

print('Done saving!')

**Congratulations on completing this assignment!**